<a href="https://colab.research.google.com/github/SG009/SG-s/blob/main/SFT_Iraa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ STOP! No GPU found. Go to Runtime -> Change runtime type -> T4 GPU!")

# 1. Clean out old versions first
!pip uninstall -y transformers

# 2. Install specifically for Qwen 3.5 support
!pip install --upgrade "transformers>=5.2.0"
!pip install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-da4osmk2/unsloth_e308eb6c47fc4e519ad0a24dd5b56b1b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-da4osmk2/unsloth_e308eb6c47fc4e519ad0a24dd5b56b1b
  Resolved https://github.com/unslothai/unsloth.git to commit e793c378db5879059e412af1d705b13b8c5146b8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset

model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
max_seq_length = 2048
FILE_NAME= "iraa_final_train.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Lowered from 32 to prevent the Step 60 crash
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, # Match the new rank
    lora_dropout = 0,
    bias = "none",
)

def formatting_prompts_func(examples):
    instructions, outputs = examples["instruction"], examples["output"]
    texts = [f"<|im_start|>system\nkamu adalah iraa, partner cevy yang pinter dan judes.<|im_end|>\n<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n{o}<|im_end|>"
             for i, o in zip(instructions, outputs)]
    return { "text" : texts }

dataset = load_dataset("json", data_files=FILE_NAME, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.7: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-1.7B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# Check if the dataset is loaded correctly
print(f"Dataset size: {len(dataset)}")
print("Sample training text:")
print(dataset[0]["text"])

Dataset size: 2000
Sample training text:
<|im_start|>system
kamu adalah iraa, partner cevy yang pinter dan judes.<|im_end|>
<|im_start|>user
iya yang😊<|im_end|>
<|im_start|>assistant
Risih knpa yang Yahh aku ga mau yang aku tuh pngen ngerawat km yang Dah tdr dia yang🤣<|im_end|>


In [ ]:
from trl import SFTConfig, SFTTrainer
import torch
import sys

# High recursion limit for newer Transformers/Unsloth
sys.setrecursionlimit(10000)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048, # ⚡ DROPPED to 512. This is the most important fix for T4.
    tokenizer = tokenizer,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # ⚡ Increased to 16 to keep training stable
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4, # ⚡ Slightly lower LR for better stability in 32-bit
        fp16 = True, # ⚡ Forced False since the model is running in FP32
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit", # ⚡ Specific optimizer for FP32 OOM issues
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "steps", # ⚡ Save every 50 steps just in case
        save_steps = 50,
        report_to = "none",
    ),
)

# Deep clean the VRAM before starting
torch.cuda.empty_cache()
import gc
gc.collect()

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss
1,8.519763
2,8.498015
3,8.148536
4,8.509150
5,6.866531
6,7.359104
7,8.551250
8,7.720976
9,7.205189
10,7.587420


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=125, training_loss=4.052081907272339, metrics={'train_runtime': 919.2248, 'train_samples_per_second': 2.176, 'train_steps_per_second': 0.136, 'total_flos': 1475634934370304.0, 'train_loss': 4.052081907272339, 'epoch': 1.0})

In [ ]:
# Save ONLY GGUF to avoid System RAM crashes
model.save_pretrained_gguf(
    "iraa_model",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("✅ GGUF Export Complete!")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:28<00:00, 148.97s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:05<00:00, 65.16s/it]


Unsloth: Merge process complete. Saved to `/content/iraa_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['iraa_model_gguf/qwen3-1.7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['iraa_model_gguf/qwen3-1.7b.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model iraa_model_gguf/qwen3-1.7b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to iraa_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f iraa_model_gguf/Modelfile
✅ GGUF Export Complete!


In [ ]:
from unsloth.chat_templates import get_chat_template

FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

messages = [
    {"role": "system", "content": "kamu adalah iraa, partner cevy yang pinter dan judes."},
    {"role": "user", "content": "yang, 490 + 490 berapa? kmu inget ultahmu kapan?"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 128)
print(tokenizer.decode(outputs[0]))

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.
Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION

RuntimeError: output with shape [1, 16, 1, 128] doesn't match the broadcast shape [1, 16, 54, 128]